# 01 · Queue: Send & Receive

Goals:

- Create a `ServiceBusClient`, `ServiceBusSender`, and `ServiceBusReceiver`
- Send a single message and a small list of messages
- Receive in **PeekLock** mode and explicitly `Complete` the message
- Contrast with **ReceiveAndDelete** mode


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

## 1. Create the client

`ServiceBusClient` is the entry point. It is **thread-safe** and meant to be a singleton in real apps.

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
Console.WriteLine($"Connected to: {client.FullyQualifiedNamespace}");

## 2. Send a single message

In [ ]:
var sender = client.CreateSender(Config.QueueName);

var message = new ServiceBusMessage("Hello, Service Bus!")
{
    MessageId = Guid.NewGuid().ToString(),
    ContentType = "text/plain"
};

await sender.SendMessageAsync(message);
Console.WriteLine($"Sent message {message.MessageId}");

## 3. Send a small list

In [ ]:
var batch = new List<ServiceBusMessage>
{
    new ServiceBusMessage("apple")  { Subject = "fruit" },
    new ServiceBusMessage("banana") { Subject = "fruit" },
    new ServiceBusMessage("carrot") { Subject = "veg"   },
};

await sender.SendMessagesAsync(batch);
Console.WriteLine($"Sent {batch.Count} messages");

## 4. Receive in PeekLock mode (the default)

In PeekLock mode the broker hands you the message but keeps a **lock** on it. You must either:

- `CompleteMessageAsync` — remove it from the queue (success)
- `AbandonMessageAsync` — release the lock so another consumer can try
- `DeadLetterMessageAsync` — move to the DLQ (poison message)
- Do nothing — the lock expires (default 30s) and the message is redelivered. `DeliveryCount` increments.


In [ ]:
var receiver = client.CreateReceiver(Config.QueueName);

for (int i = 0; i < 4; i++)
{
    var msg = await receiver.ReceiveMessageAsync(maxWaitTime: TimeSpan.FromSeconds(5));
    if (msg is null) { Console.WriteLine("(no more messages)"); break; }

    Console.WriteLine($"Received: {msg.Body}  | subject={msg.Subject}  | deliveryCount={msg.DeliveryCount}");
    await receiver.CompleteMessageAsync(msg);
}

await receiver.DisposeAsync();

## 5. ReceiveAndDelete mode

The message is deleted server-side as soon as it is delivered. **Faster, but at-most-once** — if your code crashes after receiving, the message is gone.


In [ ]:
// First, push one message we can demo with.
await sender.SendMessageAsync(new ServiceBusMessage("delete-on-receive demo"));

var fast = client.CreateReceiver(Config.QueueName, new ServiceBusReceiverOptions
{
    ReceiveMode = ServiceBusReceiveMode.ReceiveAndDelete
});

var m = await fast.ReceiveMessageAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"Got (and already deleted): {m?.Body}");

await fast.DisposeAsync();

## 6. Clean up

In [ ]:
await sender.DisposeAsync();
await client.DisposeAsync();
Console.WriteLine("Done.");

Next: [`02-message-properties.ipynb`](02-message-properties.ipynb)